In [ ]:
import os, glob, json, math, random
import numpy as np
import matplotlib.pyplot as plt

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
DATASET_PATH = "/content/drive/MyDrive/AIAGENT_FINAL/obtain_diamond_v3_fixdone"
OUT_DIR = "/content/drive/MyDrive/AIAGENT_FINAL/outputs_dataset_exploration"
PLOT_DIR = os.path.join(OUT_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

shards = sorted(glob.glob(os.path.join(DATASET_PATH, "shard_*.npz")))
print("Total shards:", len(shards))
assert len(shards) == 30, "Expected 30 shards."

Total shards: 30


In [ ]:
d = np.load(shards[0])
print("keys:", d.files)
for k in ["obs","mainhand","action","reward","done"]:
    assert k in d.files, f"Missing {k}"

print("obs", d["obs"].shape, d["obs"].dtype)
print("mainhand", d["mainhand"].shape, d["mainhand"].dtype)
print("action", d["action"].shape, d["action"].dtype)
print("reward", d["reward"].shape, d["reward"].dtype)
print("done", d["done"].shape, d["done"].dtype)

state = np.concatenate([d["obs"].astype(np.float32), d["mainhand"].astype(np.float32)], axis=1)
print("state", state.shape, state.dtype)
assert state.shape[1] == 28
assert d["action"].shape[1] == 15

keys: ['obs', 'mainhand', 'action', 'reward', 'done']
obs (66160, 20) float32
mainhand (66160, 8) uint8
action (66160, 15) float32
reward (66160,) float32
done (66160,) bool
state (66160, 28) float32


In [ ]:
def shard_episode_lengths(done_arr: np.ndarray) -> list[int]:
    """
    done_arr: shape (N,) bool
    Each shard may contain multiple episodes; episodes end where done==True.
    Returns list of episode lengths within the shard.
    """
    done_idx = np.where(done_arr)[0]
    if len(done_idx) == 0:
        return [len(done_arr)]
    lens = []
    start = 0
    for end in done_idx:
        lens.append((end - start) + 1)
        start = end + 1
    if start < len(done_arr):
        lens.append(len(done_arr) - start)
    return lens

def action_binary_rates(action_mat: np.ndarray, binary_dims=8) -> np.ndarray:
    """
    First 8 are movement/attack-like binaries per your spec.
    Returns rate per dim in [0,1].
    """
    x = action_mat[:, :binary_dims]
    # tolerate float actions that are {0,1}
    x = (x > 0.5).astype(np.float32)
    return x.mean(axis=0)

def camera_stats(action_mat: np.ndarray, cam_idx=(8, 9)) -> dict:
    cam = action_mat[:, list(cam_idx)]
    return {
        "camera_mean": cam.mean(axis=0).tolist(),
        "camera_std": cam.std(axis=0).tolist(),
        "camera_p50": np.median(cam, axis=0).tolist(),
        "camera_p95_abs": np.percentile(np.abs(cam), 95, axis=0).tolist(),
    }

def reward_summary(rew: np.ndarray) -> dict:
    rew = rew.astype(np.float32)
    nz = rew[rew != 0]
    return {
        "reward_min": float(rew.min()),
        "reward_max": float(rew.max()),
        "reward_mean": float(rew.mean()),
        "reward_std": float(rew.std()),
        "reward_nonzero_rate": float((rew != 0).mean()),
        "reward_nonzero_count": int((rew != 0).sum()),
        "reward_nonzero_unique_top20": np.unique(nz)[:20].tolist() if nz.size else [],
    }

In [ ]:
agg = {
    "n_shards": len(shards),
    "shards": [],
    "total_steps": 0,
    "total_episodes": 0,
    "episode_lengths": [],
    "reward_nonzero_total": 0,
    "done_true_total": 0,
    "action_binary_rate_sum": np.zeros(8, dtype=np.float64),
    "action_binary_rate_weight": 0.0,
    "camera_means": [],
    "camera_stds": [],
    "state_feature_mean_sum": np.zeros(28, dtype=np.float64),
    "state_feature_var_sum": np.zeros(28, dtype=np.float64),
    "state_feature_count": 0,
}

for p in shards:
    d = np.load(p)
    obs = d["obs"].astype(np.float32)
    mainhand = d["mainhand"].astype(np.float32)
    action = d["action"].astype(np.float32)
    reward = d["reward"].astype(np.float32)
    done = d["done"].astype(bool)

    N = obs.shape[0]
    agg["total_steps"] += N
    agg["done_true_total"] += int(done.sum())

    # episode lengths
    lens = shard_episode_lengths(done)
    agg["episode_lengths"].extend(lens)
    agg["total_episodes"] += len(lens)

    # reward stats per shard (store light summary)
    rs = reward_summary(reward)
    agg["reward_nonzero_total"] += rs["reward_nonzero_count"]

    # action binary rates (weighted by steps)
    br = action_binary_rates(action, binary_dims=8)
    agg["action_binary_rate_sum"] += br * N
    agg["action_binary_rate_weight"] += N

    # camera stats (store list; aggregate later)
    cs = camera_stats(action, cam_idx=(8, 9))
    agg["camera_means"].append(cs["camera_mean"])
    agg["camera_stds"].append(cs["camera_std"])

    # state stats (mean/var streaming)
    state = np.concatenate([obs, mainhand], axis=1)
    agg["state_feature_mean_sum"] += state.sum(axis=0)
    agg["state_feature_var_sum"] += (state ** 2).sum(axis=0)
    agg["state_feature_count"] += N

    agg["shards"].append({
        "name": os.path.basename(p),
        "steps": int(N),
        "episodes_in_shard": len(lens),
        "reward_nonzero_rate": rs["reward_nonzero_rate"],
        "reward_max": rs["reward_max"],
        "done_true": int(done.sum()),
    })

# finalize aggregates
ep = np.array(agg["episode_lengths"], dtype=np.int64)
agg["episode_length_mean"] = float(ep.mean()) if ep.size else None
agg["episode_length_std"] = float(ep.std()) if ep.size else None
agg["episode_length_p50"] = float(np.median(ep)) if ep.size else None
agg["episode_length_p90"] = float(np.percentile(ep, 90)) if ep.size else None
agg["episode_length_p99"] = float(np.percentile(ep, 99)) if ep.size else None
agg["reward_nonzero_rate_overall"] = float(agg["reward_nonzero_total"] / max(agg["total_steps"], 1))
agg["done_true_rate_overall"] = float(agg["done_true_total"] / max(agg["total_steps"], 1))

agg["action_binary_rate"] = (agg["action_binary_rate_sum"] / max(agg["action_binary_rate_weight"], 1.0)).tolist()

cam_means = np.array(agg["camera_means"], dtype=np.float32)
cam_stds = np.array(agg["camera_stds"], dtype=np.float32)
agg["camera_mean_over_shards"] = cam_means.mean(axis=0).tolist()
agg["camera_std_over_shards"] = cam_stds.mean(axis=0).tolist()

cnt = max(agg["state_feature_count"], 1)
mu = agg["state_feature_mean_sum"] / cnt
mu2 = agg["state_feature_var_sum"] / cnt
var = np.maximum(mu2 - mu**2, 0.0)
agg["state_feature_mean"] = mu.tolist()
agg["state_feature_std"] = np.sqrt(var).tolist()

print("Total steps:", agg["total_steps"])
print("Total episodes:", agg["total_episodes"])
print("Reward nonzero rate overall:", agg["reward_nonzero_rate_overall"])
print("Done true rate overall:", agg["done_true_rate_overall"])
print("Action binary rates (8):", agg["action_binary_rate"])

Total steps: 1916597
Total episodes: 122
Reward nonzero rate overall: 0.0006652415713892905
Done true rate overall: 6.365448761528898e-05
Action binary rates (8): [0.18897608728028115, 0.02743195382061462, 0.011046662343960293, 0.024036351914086488, 0.03396019102823422, 0.04068721808406025, 0.018301186883799614, 0.5020335501379528]


In [ ]:
import json
import numpy as np

def to_jsonable(x):
    # numpy scalars
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, (np.bool_,)):
        return bool(x)
    # numpy arrays
    if isinstance(x, np.ndarray):
        return x.tolist()
    # containers
    if isinstance(x, dict):
        return {str(k): to_jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [to_jsonable(v) for v in x]
    # passthrough
    return x

summary_path = os.path.join(OUT_DIR, "dataset_summary.json")
with open(summary_path, "w") as f:
    json.dump(to_jsonable(agg), f, indent=2)

print("Wrote:", summary_path)
print("Plots in:", PLOT_DIR)

Wrote: /content/drive/MyDrive/AIAGENT_FINAL/outputs_dataset_exploration/dataset_summary.json
Plots in: /content/drive/MyDrive/AIAGENT_FINAL/outputs_dataset_exploration/plots
